## Introduction to Machine Learning

In [36]:
%%duckdb

select id, title,score  from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null
order by score desc
limit 10

,id,title,score
0,7566069,Drop Dropbox,1990
1,7548991,The Heartbleed Bug,1768
2,21923220,"IRS Reforms Free File Program, Drops Agreement...",1721
3,14754772,Net Neutrality Day of Action: Help Preserve th...,1664
4,11166417,Stripe Atlas,1659
5,21942358,Which Emoji Scissors Close?,1545
6,21998638,Broot – A new way to see and navigate director...,1448
7,14737322,Taking control of all .io domains with a targe...,1404
8,11218677,Dsxyliea,1356
9,21961560,Ask HN: I've been slacking off at Google for 6...,1337


### Simple Regression wrt Number of Tokens (aka Title Length)

In [17]:
%%duckdb

select regr_intercept(score,x ), regr_slope(score, x) 
from
(select array_length(string_split(title, ' ')) x, score  from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null)

,"regr_intercept(score, x)","regr_slope(score, x)"
0,10.521204,-0.059761


In [37]:
%%duckdb 

with result as (select id, title, -0.059761* array_length(string_split(title, ' ')) +10.521204 score_pred, score 
from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null)
select median(abs(score_pred - score)) mae, sqrt(avg(pow(score_pred - score,2))) rmse  from result

,mae,rmse
0,8.804072,42.974893


### Word based

In [75]:
%%duckdb


create or replace table word_statistics as 
select word, avg(score) mean_word_score, count(distinct id) doc_freq from
(select id, score, unnest(list_transform(string_split(lower(title),' '), lambda w: trim(w,',.?():'))) word  
from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null) 
group by 1
having count(distinct id) > 10
order by 2 desc;

create or replace table best50_words as select * from word_statistics order by mean_word_score desc limit 50;

create or replace table worst50_words as  select * from word_statistics order by mean_word_score  limit 50;


select id, max(score) score, mean(ifnull(mean_word_score,0)) score_pred from 
(select id, score, unnest(list_transform(string_split(lower(title),' '), lambda w: trim(w,',.?():'))) word  
from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null) f LEFT JOIN word_statistics d USING(word) group by 1

,id,score,score_pred
0,361717,3,9.129567
1,361719,1,7.664272
2,361721,1,11.709595
3,361724,27,8.350148
4,361726,9,7.034539
...,...,...,...
147586,4042950,2,4.303814
147587,4042952,1,9.378855
147588,4042957,28,10.021164
147589,14827603,1,0.000000


In [76]:
%%duckdb 

with result as (select id, max(score) score, mean(ifnull(mean_word_score,0)) score_pred from 
(select id, score, unnest(list_transform(string_split(lower(title),' '), lambda w: trim(w,',.?():'))) word  
from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null) f LEFT JOIN word_statistics d USING(word) group by 1)
select median(abs(score_pred - score)) mae, sqrt(avg(pow(score_pred - score,2))) rmse  from result

,mae,rmse
0,6.682482,42.485492


### Similarity Based

In [97]:
%%duckdb 

with sample as
(select id, score, list_transform(string_split(lower(title),' '), lambda w: trim(w,',.?():')) word  
from 's3://tt-bootcamp-shared/hackernews.parquet' 
where title is not null and score is not null limit 3000),
result as (select id, max(score) as score, avg(other_score) as score_pred from
(select *, row_number() over(partition by id order by jaccard_sim desc) rn from 
(select f.*, list_unique(list_intersect(f.word, o.word))/list_unique( list_concat(f.word, o.word)) jaccard_sim, o.score other_score
from sample f  join sample o on (f.id != o.id)) QUALIFY rn <= 3) group by 1)
select median(abs(score_pred - score)) mae, sqrt(avg(pow(score_pred - score,2))) rmse  from result

,mae,rmse
0,3.333333,16.107357


- a - 0
- b - 1
- c - 2

{a,b} {b,c}

$<1 1 0>$
$<0 1 1>$

$Cosine -> 1$

$ 1 / 3 $